# 🔴 OpenSource Clipping — Live Stream Clipper (Kaggle / Colab)

Clips a stream **while it is live**: records continuously, and every few minutes the AI reads the latest stretch of transcript and renders the best moment as a vertical short.

**Before running (Kaggle):**
1. **Settings → Accelerator → GPU T4 x2** (or P100), and **Internet → On** (needs a phone-verified Kaggle account).
2. **Add-ons → Secrets** → add `GOOGLE_API_KEY` and tick it for this notebook. Optional: `HF_TOKEN`, `PEXELS_API_KEY`, `NVIDIA_API_KEY`.
3. **YouTube cookies (usually required on Kaggle):** YouTube often blocks datacenter IPs with a "confirm you're not a bot" check. Export `youtube_cookies.txt` (Netscape format) from a browser logged in to a **spare** Google account, upload it as a **private** Kaggle Dataset, and add that dataset to this notebook. It is copied to a temporary folder and never saved to the notebook output.
4. Fill in **Settings** below, then **Save Version → Save & Run All (Commit)** so it keeps running with the browser closed (max 12 h).

Clips are written straight to `/kaggle/working/outputs/live/…`, so they appear in the version's **Output** tab even if the session is cut off. Keep this notebook **private**.

## 1. Settings

In [ ]:
STREAM_URL = ""          # e.g. "https://www.youtube.com/@SomeStreamer/live" or a /watch?v=... link of a live stream
CONTENT_PROFILE = "gaming"   # "gaming" or "podcast"
RATIO = "9:16"
SOURCE_HEIGHT = "1080"   # max recording height ("max", "1080", "720"...)
LAYOUT = "gameplay"      # "gameplay" = whole game screen + title on top + captions below; "crop" = face-tracked crop
FONT_STYLE = "GTA"       # caption font preset: "GTA", "HORMOZI", "DEFAULT", "CINEMATIC", "STORYTELLER"

LIVE_WINDOW = 600        # seconds of recent stream the AI reads each time
LIVE_STEP = 300          # analyse a new window every N seconds (keep WINDOW - STEP >= 180)
LIVE_MIN_SCORE = 80      # only render clips the AI scores >= this (1-100)
LIVE_CLIPS_PER_WINDOW = 1
LIVE_MAX_HOURS = 11.5    # Kaggle stops sessions at 12 h; stop cleanly before that

WHISPER_MODEL = "large-v3"   # "medium" / "small" if the log says it is falling behind
GEMINI_MODEL = "gemini-3.5-flash"       # stable model; preview models are often overloaded (503)
GEMINI_FALLBACK_MODEL = "gemini-3.1-flash-lite"   # fast backup when the main model is busy
EXTRA_ARGS = []              # any other main.py flags, e.g. ["--no-bgm", "--font-style", "bold"]

REPO_URL = "https://github.com/BilalShaikh1444/opensource-clipping.git"
REPO_BRANCH = "feat/live-mode"

if not STREAM_URL.strip():
    raise ValueError("Set STREAM_URL in this cell first.")
print("Settings OK.")

## 2. Setup

Code and the Python environment go to a temporary folder (not saved with the notebook). Only `outputs/` is linked into `/kaggle/working`. Rerunning updates the checkout to the latest commit of `REPO_BRANCH`. The first setup takes several minutes.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def _require(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            "Run the earlier cells first, top to bottom (or click Run All). "
            f"Missing: {', '.join(missing)}. A restarted session also needs everything rerun."
        )

_require("REPO_URL", "REPO_BRANCH")

ON_KAGGLE = Path("/kaggle/working").is_dir()
ON_COLAB = Path("/content").is_dir() and not ON_KAGGLE
SAVED_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if ON_COLAB else Path.cwd()
PROJECT_DIR = Path("/tmp/opensource-clipping") if (ON_KAGGLE or ON_COLAB) else SAVED_DIR / "opensource-clipping"

if not (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=PROJECT_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=PROJECT_DIR, check=True)
    subprocess.run(["git", "merge", "--ff-only", f"origin/{REPO_BRANCH}"], cwd=PROJECT_DIR, check=True)
commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], cwd=PROJECT_DIR,
                        capture_output=True, text=True, check=True).stdout.strip()
if not (PROJECT_DIR / "clipping" / "live.py").is_file():
    raise RuntimeError(f"{REPO_BRANCH} does not contain live mode (clipping/live.py). Check REPO_URL / REPO_BRANCH.")

# Clips go straight to the saved output folder.
OUTPUTS_DIR = SAVED_DIR / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
link = PROJECT_DIR / "outputs"
if link.is_symlink() or not link.exists():
    if link.is_symlink():
        link.unlink()
    link.symlink_to(OUTPUTS_DIR, target_is_directory=True)
elif link.resolve() != OUTPUTS_DIR.resolve():
    raise RuntimeError(f"{link} is a real folder; move or delete it, then rerun this cell.")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print(f"Checkout: {commit}\nCode: {PROJECT_DIR}\nClips: {OUTPUTS_DIR}")

In [ ]:
# Isolated, version-pinned environment (same builder as the Studio notebook): CUDA torch, Whisper, MediaPipe, Deno for yt-dlp.
import subprocess
import sys

def _require(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            "Run the earlier cells first, top to bottom (or click Run All). "
            f"Missing: {', '.join(missing)}. A restarted session also needs everything rerun."
        )

_require("PROJECT_DIR")
import hashlib
import os
import shutil
import tarfile
import urllib.request
from pathlib import Path

# The pinned libraries (torch 2.5.1, MediaPipe 0.10.21) support Python 3.10-3.12 only.
# Newer Kaggle images run Python 3.13, so fetch a standalone Python 3.12 with uv
# (pinned release, checksum-verified) and build the clipping environment from it.
UV_VERSION = "0.8.0"
UV_SHA256 = "a7d74ee5c5ff3069b9d88236a05f293cc4e2809bad872f3a88a384489ba3675e"
TOOLS_DIR = Path("/tmp/clipping-tools")


def python_312_or_older():
    if (3, 10) <= sys.version_info[:2] <= (3, 12):
        return sys.executable
    found = shutil.which("python3.12")
    if found:
        return found
    uv = TOOLS_DIR / "uv"
    if not uv.is_file():
        TOOLS_DIR.mkdir(parents=True, exist_ok=True)
        url = f"https://github.com/astral-sh/uv/releases/download/{UV_VERSION}/uv-x86_64-unknown-linux-gnu.tar.gz"
        archive = TOOLS_DIR / "uv.tar.gz"
        print(f"Kernel is Python {sys.version_info.major}.{sys.version_info.minor}; downloading uv {UV_VERSION} to get Python 3.12...")
        urllib.request.urlretrieve(url, archive)
        if hashlib.sha256(archive.read_bytes()).hexdigest() != UV_SHA256:
            archive.unlink()
            raise RuntimeError("uv download checksum mismatch; nothing was installed.")
        with tarfile.open(archive) as tar:
            member = tar.getmember("uv-x86_64-unknown-linux-gnu/uv")  # extract only the binary
            with tar.extractfile(member) as src, open(uv, "wb") as dst:
                shutil.copyfileobj(src, dst)
        uv.chmod(0o755)
        archive.unlink()
    uv_env = {**os.environ, "UV_PYTHON_INSTALL_DIR": str(TOOLS_DIR / "python")}
    subprocess.run([str(uv), "python", "install", "3.12"], env=uv_env, check=True)
    return subprocess.run([str(uv), "python", "find", "3.12"], env=uv_env,
                          capture_output=True, text=True, check=True).stdout.strip()


BUILD_PYTHON = python_312_or_older()
print("Building the clipping environment with", subprocess.run(
    [BUILD_PYTHON, "--version"], capture_output=True, text=True, check=True).stdout.strip())

ENV_DIR = PROJECT_DIR / ".cache" / "kaggle-studio" / "venv"
ENV_PYTHON = ENV_DIR / "bin" / "python"
# Drop an environment left over from a failed run with the wrong Python version.
if ENV_PYTHON.exists():
    probe = subprocess.run([str(ENV_PYTHON), "-c", "import sys; print(sys.version_info[:2] <= (3, 12))"],
                           capture_output=True, text=True)
    if probe.stdout.strip() != "True":
        shutil.rmtree(ENV_DIR)
subprocess.run([BUILD_PYTHON, "scripts/kaggle_studio.py", "setup", "--venv", str(ENV_DIR)], check=True)

if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL)
print("Environment and FFmpeg ready.")

## 3. Secrets and cookies

Values are passed to the clipper through the process environment only; they are never printed or written to disk.

In [ ]:
import os

def _require(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            "Run the earlier cells first, top to bottom (or click Run All). "
            f"Missing: {', '.join(missing)}. A restarted session also needs everything rerun."
        )

_require("PROJECT_DIR")
readers = []
try:
    from kaggle_secrets import UserSecretsClient
    readers.append(UserSecretsClient().get_secret)
except ImportError:
    pass
except Exception as exc:
    print(f"Kaggle Secrets unavailable ({type(exc).__name__}).")
try:
    from google.colab import userdata
    readers.append(userdata.get)
except ImportError:
    pass

for name in ("GOOGLE_API_KEY", "HF_TOKEN", "PEXELS_API_KEY", "NVIDIA_API_KEY"):
    value = os.environ.get(name, "").strip()
    for read in readers:
        if value:
            break
        try:
            value = (read(name) or "").strip()
        except Exception:
            pass
    if value:
        os.environ[name] = value
    print(f"{name}: {'configured' if value else 'not configured'}")
value = None

if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError("GOOGLE_API_KEY is missing: add it under Add-ons → Secrets and tick it for this notebook, then rerun.")

In [ ]:
# Find youtube_cookies.txt in attached datasets (or /content on Colab) and copy it next to the code (not saved output).
import glob
import os
import shutil
from pathlib import Path

def _require(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            "Run the earlier cells first, top to bottom (or click Run All). "
            f"Missing: {', '.join(missing)}. A restarted session also needs everything rerun."
        )

_require("PROJECT_DIR")
COOKIES = None
candidates = sorted(glob.glob("/kaggle/input/**/*cookies*.txt", recursive=True)) + ["/content/youtube_cookies.txt"]
source = next((p for p in candidates if os.path.isfile(p)), None)
if source:
    COOKIES = str(PROJECT_DIR / "youtube_cookies.txt")
    shutil.copyfile(source, COOKIES)
    os.chmod(COOKIES, 0o600)
    print(f"Cookies: using {Path(source).name} from an attached dataset.")
else:
    print("No cookies file found. If the stream check below says 'not a bot', attach your private cookies dataset and rerun.")

## 4. Pre-flight checks

Confirms the GPU is visible to Whisper and that the stream is live and readable, before the long run starts.

In [ ]:
import subprocess

def _require(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            "Run the earlier cells first, top to bottom (or click Run All). "
            f"Missing: {', '.join(missing)}. A restarted session also needs everything rerun."
        )

_require("STREAM_URL", "SOURCE_HEIGHT", "PROJECT_DIR", "ENV_DIR", "ENV_PYTHON", "COOKIES")
if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError("GOOGLE_API_KEY is not loaded: run the Secrets cell first.")
from scripts.kaggle_studio import runtime_environment
RUN_ENV = runtime_environment(ENV_DIR)

check = r'''
import sys, torch, ctranslate2
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-",
      "| CTranslate2 CUDA devices:", ctranslate2.get_cuda_device_count())
from clipping import live
info = live.resolve_live_stream(sys.argv[1], sys.argv[2] or None, sys.argv[3])
print("Stream:", info["live_status"], "|", info["title"][:80], "|", len(info["inputs"]), "input(s)")
if not info["is_live"]:
    sys.exit("Stream is not live right now. Use a live URL (or run the normal pipeline for a finished stream).")
'''
result = subprocess.run([str(ENV_PYTHON), "-c", check, STREAM_URL, COOKIES or "", SOURCE_HEIGHT],
                        cwd=PROJECT_DIR, env=RUN_ENV)
if result.returncode != 0:
    raise RuntimeError("Pre-flight failed (see above). Typical fixes: enable the GPU accelerator, attach cookies, check the URL.")

USE_GPU = "CUDA: True" in subprocess.run(
    [str(ENV_PYTHON), "-c", "import torch; print('CUDA:', torch.cuda.is_available())"],
    cwd=PROJECT_DIR, env=RUN_ENV, capture_output=True, text=True).stdout
if not USE_GPU:
    print("⚠️ No GPU: Whisper will run on CPU with a smaller model and will likely fall behind. Enable the GPU accelerator.")

## 5. Run live clipping

Runs until the stream ends, `LIVE_MAX_HOURS` passes, or you stop the cell (⏹). Stopping saves the manifest and keeps every finished clip. Each window logs how far behind live it is.

In [ ]:
import signal
import subprocess

def _require(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            "Run the earlier cells first, top to bottom (or click Run All). "
            f"Missing: {', '.join(missing)}. A restarted session also needs everything rerun."
        )

_require("STREAM_URL", "CONTENT_PROFILE", "RATIO", "SOURCE_HEIGHT", "LIVE_WINDOW", "LIVE_STEP",
         "LIVE_MIN_SCORE", "LIVE_CLIPS_PER_WINDOW", "LIVE_MAX_HOURS", "WHISPER_MODEL", "EXTRA_ARGS",
         "GEMINI_MODEL", "GEMINI_FALLBACK_MODEL", "LAYOUT", "FONT_STYLE",
         "PROJECT_DIR", "ENV_PYTHON", "COOKIES", "RUN_ENV", "USE_GPU")

cmd = [
    str(ENV_PYTHON), "-u", "main.py",
    "--url", STREAM_URL, "--live",
    "--content-profile", CONTENT_PROFILE,
    "--ratio", RATIO,
    "--source-height", str(SOURCE_HEIGHT),
    "--live-window", str(LIVE_WINDOW),
    "--live-step", str(LIVE_STEP),
    "--live-min-score", str(LIVE_MIN_SCORE),
    "--live-clips-per-window", str(LIVE_CLIPS_PER_WINDOW),
    "--live-max-hours", str(LIVE_MAX_HOURS),
    "--gemini-model", GEMINI_MODEL,
    "--gemini-fallback-model", GEMINI_FALLBACK_MODEL,
    "--layout", LAYOUT,
    "--font-style", FONT_STYLE,
]
if USE_GPU:
    cmd += ["--whisper-model", WHISPER_MODEL, "--whisper-device", "cuda", "--whisper-compute-type", "float16"]
else:
    cmd += ["--whisper-model", "small", "--whisper-device", "cpu", "--whisper-compute-type", "int8"]
if COOKIES:
    cmd += ["--yt-cookies", COOKIES]
cmd += list(EXTRA_ARGS)

proc = subprocess.Popen(cmd, cwd=PROJECT_DIR, env=RUN_ENV, start_new_session=True)
try:
    code_ = proc.wait()
    print(f"\nClipper exited with code {code_}.")
except KeyboardInterrupt:
    print("\nStopping: letting the clipper save its manifest...")
    proc.send_signal(signal.SIGINT)
    try:
        proc.wait(timeout=120)
    except subprocess.TimeoutExpired:
        proc.terminate()
        proc.wait(timeout=30)

## 6. Results

In [ ]:
import json
import os
import zipfile
from pathlib import Path

def _require(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            "Run the earlier cells first, top to bottom (or click Run All). "
            f"Missing: {', '.join(missing)}. A restarted session also needs everything rerun."
        )

_require("OUTPUTS_DIR", "SAVED_DIR")

runs = sorted((OUTPUTS_DIR / "live").glob("*/live_manifest.json"))
if not runs:
    print("No live runs yet.")
else:
    manifest_path = runs[-1]
    rows = json.loads(manifest_path.read_text(encoding="utf-8"))
    ok = [r for r in rows if r.get("status") == "success" and r.get("video_path") and os.path.exists(r["video_path"])]
    print(f"Run: {manifest_path.parent.name} — {len(ok)} clip(s) ready, {len(rows) - len(ok)} failed/missing\n")
    for r in rows:
        s = int(r.get("live_stream_start") or 0)
        print(f"[{s // 3600}:{s % 3600 // 60:02d}:{s % 60:02d}] {r.get('status'):8} {r.get('youtube_title_final', '')}")
        print(f"      {r.get('video_path')}")
    zip_path = SAVED_DIR / f"live_clips_{manifest_path.parent.name}.zip"
    with zipfile.ZipFile(zip_path, "w") as zf:
        zf.write(manifest_path, "live_manifest.json")
        for r in ok:
            zf.write(r["video_path"], Path(r["video_path"]).parent.name + "_" + Path(r["video_path"]).name)
            thumb = r.get("thumbnail_path")
            if thumb and os.path.exists(thumb):
                zf.write(thumb, Path(thumb).parent.name + "_" + Path(thumb).name)
    print(f"\nAll clips zipped: {zip_path} (download from the Output tab).")